In [1]:
import os
import pandas as pd
import numpy as np
import re
import pandas as pd
import plotly.express as px
import simplejson as json
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import ast
from collections import Counter, defaultdict

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
base_dir = "/content/drive/MyDrive/DSCI_550/assignment_3/JSON_Data"
file_path = os.path.join(base_dir, "datarandomsample.json")

Visualization #1: Bubble Map

In [4]:
# Load and prepare data
#data_path = 'https://github.com/kaavyak/DSCI550_Assignment3/blob/main/datanonbw.json'
df = pd.read_json(file_path, lines=True)

df['SightingsCount'] = df.groupby(['State'])['State'].transform('count')

# Construct a radius scale based on SightingsCount, similar to the D3.js example
# Here, we'll use the size parameter of scatter_geo to simulate this effect
max_sightings = df['SightingsCount'].max()
df['ScaledSightings'] = df['SightingsCount'] / max_sightings * 40  # Scale sightings

fig = px.scatter_geo(df,
                     lat='latitude',
                     lon='longitude',
                     size='SightingsCount',
                     hover_name='city',
                     hover_data=['State', 'SightingsCount'],
                     title='Geographical Distribution of Haunted Places Across the U.S.',
                     scope='usa',  # Automatically focuses the map on the USA
                     projection='albers usa',  # Uses the Albers projection, common for US maps
                     color='SightingsCount',
                     color_continuous_scale=px.colors.sequential.Plasma)
fig.update_geos(
    landcolor='Gray',
    oceancolor='LightBlue',
    showocean=True,
    lakecolor='LightBlue'
)
fig.show()
fig.write_html("hauntedplaces_heatmap.html")

Visualization #2: Stacked Bar Chart

In [5]:
# Load the dataset and create the DataFrame
df = pd.read_json(file_path, lines=True)

# Group by Time of Day and Apparition Type, then count the sightings
pivot_df = df.groupby(['time_of_day', 'apparition_type']).size().unstack(fill_value=0)

# Continue with the visualization...

total_sightings_by_time = df.groupby('time_of_day').size().reset_index(name='TotalSightings')

# Define contrasting colors for the apparition type categories
contrasting_colors = ['#17becf', '#bcbd22', '#7f7f7f', '#e377c2', '#41c631', '#3148c6', '#c631c2', '#f2f78f', '#8b451d', '#ccefff', '#2f1283']

# Create a figure with secondary y-axis for the bar and line chart
fig = make_subplots(specs=[[{"secondary_y": True}]])

# Add a bar trace for each apparition type category in the pivoted DataFrame with contrasting colors
for index, category in enumerate(pivot_df.columns):
    fig.add_trace(
        go.Bar(x=pivot_df.index, y=pivot_df[category], name=category,
               marker_color=contrasting_colors[index % len(contrasting_colors)])
    )

# Update layout for stacked bar chart and line trace
fig.update_layout(
    barmode='stack',
    title='Haunted Place Sightings Over Time of Day by Apparition Type with Total Count',
    xaxis=dict(showline=False, showgrid=False, showticklabels=True,
               linecolor='rgb(204, 204, 204)', linewidth=2,
               ticks='outside', tickfont=dict(family='Arial', size=12, color='rgb(82, 82, 82)')),
    yaxis=dict(title='Number of Sightings', showgrid=False, zeroline=False, showline=False, showticklabels=True),
    plot_bgcolor='white',
    autosize=True,
    margin=dict(autoexpand=True, l=100, r=20, t=110),
    showlegend=True,
)

# Customize the axes to match the D3.js example as close as possible
fig.update_xaxes(title_text='Time of Day')
fig.update_yaxes(title_text='Number of Sightings', secondary_y=False)

# Show the figure
fig.show()
fig.write_html("sightings_time_apparition_bargraph.html")

In [6]:
df.iloc[:,13:23]

,city_latitude,audio_evidence,image_video_visual_evidence,apparition_type,time_of_day,event_type,cleaned_description,witness_count,haunted_places_date,total_deaths
0,42.601119,False,True,Shadow Figure,Night,Unknown,old civil war mansion was part of the undergr...,Unknown,2025-01-01T00:00:00Z,2651
1,34.014264,False,False,Unknown,Unknown,Unknown,this middle school is supposedly haunted by ge...,Unknown,2025-01-01T00:00:00Z,2208
2,21.501050,False,False,Ghost,Afternoon,Unknown,at wahiawa middle school there is said to be a...,Unknown,2025-10-14T00:00:00Z,526
3,39.099727,False,False,Unknown,Night,Illness,on a rainy night there is a headless woman tha...,Unknown,2025-01-01T00:00:00Z,2877
4,38.958231,True,True,Spectral Apparition,Night,Murder,fiddlers green a pioneer woman who was said t...,Unknown,2025-01-01T00:00:00Z,15443
...,...,...,...,...,...,...,...,...,...,...
3495,36.122311,False,True,Shadow Figure,Night,Unknown,said that a girl is seen walking around the at...,Unknown,2025-01-01T00:00:00Z,3359
3496,41.140836,False,True,Unknown,Night,Unknown,all that has been seen so far is a picture in ...,Unknown,2025-01-01T00:00:00Z,1426
3497,38.833882,True,False,Victim Apparition,Afternoon,Illness,about 40 years ago two 3rd graders committed s...,40,2040-03-14T00:00:00Z,2623
3498,32.241267,True,True,Unknown,Unknown,Unknown,walker rd surrounding area witnesses have see...,Unknown,2025-01-01T00:00:00Z,2278


Visualization #3: Treemap

In [7]:
#Visualization 3
#Visualizing the density of audio evidence and image evidence per state in a treemap

#Answers question from HW1:
#Is there a set of frequently co-occurring features that define a particular Haunted Place?
#co-occuring features: audio, image and state

#treemap: the size of the rectangle representes the size (count) of each evidence and is divided into each state

#Loading in the data
# df = pd.read_json(file_path, lines=True)

def evidence(row):
  if row["audio_evidence"] and row["image_video_visual_evidence"]:
    return "Audio and Image Evidence"
  elif row["audio_evidence"]:
    return "Audio Evidence"
  elif row["image_video_visual_evidence"]:
    return "Image Evidence"
  else:
    return "No Evidence"

df["evidence"] = df.apply(evidence, axis=1)

#Prepare the grouped data for the plotly
grouped_df = df.groupby(["state","evidence"]).size().reset_index(name="count")

#treemap
fig = px.treemap(grouped_df, path=["state", "evidence"], values="count",color="evidence",title="Type of Evidence by State")
# add a legend
fig.update_layout(legend_title_text="type",legend=dict(orientation="v", yanchor="bottom", y=1.02, xanchor="right", x=1))

fig.show()
fig.write_html("/content/drive/MyDrive/DSCI_550/assignment_3/evidence_treemap.html")


Visualization #4: Donut Pie Chat

In [8]:
#Answers questions from HW2:
#Are there any specific trends you see in the text captions or identified objects in the image media?

# df = pd.read_json(file_path, lines=True)
df["detected_objects_classnames"] = df["detected_objects_classnames"].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
#dropping na values
df = df[df["detected_objects_classnames"].notna()]
exploded_df = df.explode("detected_objects_classnames")
exploded_counts = exploded_df["detected_objects_classnames"].value_counts().reset_index()
exploded_counts.columns = ["object_name", "count"]

fig = px.pie(
    values=exploded_counts["count"],
    names=exploded_counts["object_name"],
    hole=0.3,
    title="Distribution of Detected Objects",
)
fig.update_traces(textinfo="none", hoverinfo="label+value+percent")
fig.show()
fig.write_html("/content/drive/MyDrive/DSCI_550/assignment_3/detected_objects_treemap.html")

Visualization #5: Sankey Diagram by Riya Berry

In [9]:
# Answers question from hw 2 question: Do the Entities provide any further context about the Haunted place?
# Related: Is there a correlation between location (US region) of where the haunted event occured, and entities discussed in the description?

# Convert entities to type: list (currently a string)
df['entities'] = df['spacy_entities'].apply(ast.literal_eval)

# Count all entity names in relevant categories (exclude SPACY ordinal, cardinal, date, time categories)
valid_types = {"PERSON", "ORG", "GPE", "LOC", "EVENT"}

entity_names = []
for entity_dict in df['entities']:
    for ent_type, name_list in entity_dict.items():
        if ent_type in valid_types:
            entity_names += name_list

# get the top 20 most frequently occuring entities from all descriptions (pre-processing each entity)
top_N = 20
top_entities = set(name.lower().replace('the', '').strip() for name, _ in Counter(entity_names).most_common(top_N))
top_entities

{'alex',
 'auditorium',
 'civil war',
 'confederate',
 'elizabeth',
 'george',
 'house',
 'inn',
 'joe',
 'mary',
 'nasa',
 'ohio',
 'rumor',
 'sarah',
 'tb',
 'tommy',
 'update/correction',
 'wwii'}

In [10]:
# for each row, create a column that indicates which of the top 20 SpaCy entities a given description contains
def generate_entity_column(df):
  df['final_entities'] = None

  for index, row in df.iterrows():
    final_entities = {}
    for ent_type, ent_list in row['entities'].items():
        for ent_name in ent_list:
          # pre-processing the entity name
            ent_name = ent_name.lower().replace('the', '').strip()
            # ensure entity is one of the top 20 entities
            if ent_name in top_entities:
              final_entities[ent_type] = ent_name

    # update the original dataframe
    df.at[index, 'final_entities'] = final_entities

  return df

In [11]:
new_df = generate_entity_column(df)
new_df['final_entities'].value_counts()

,count
final_entities,
{},3303
{'EVENT': 'civil war'},35
{'PERSON': 'rumor'},27
{'PERSON': 'inn'},18
{'PERSON': 'mary'},10
{'EVENT': 'wwii'},10
{'ORG': 'tb'},10
{'ORG': 'house'},8
{'ORG': 'update/correction'},8


In [12]:
# Create node labels for Sankey diagram
labels = []
source = []
target = []
values = []

# get the index location of a label
def get_index(label):
    if label not in labels:
        labels.append(label)
    return labels.index(label)

# count the links between nodes
link_counter = defaultdict(int)

# Build links: Place --> SpACy Entity Category --> Entity Name
for index, row in new_df.iterrows():
    place = row['US_region']
    entity_dict = row["final_entities"]

    for ent_type, ent_name in entity_dict.items():
        # Place --> SpaCy Entity Category
        src1 = get_index(place)
        tgt1 = get_index(ent_type)
        link_counter[(src1, tgt1)] += 1

        # SpaCy Entity Category --> Entity Name
        src2 = tgt1
        tgt2 = get_index(ent_name)
        link_counter[(src2, tgt2)] += 1

# Convert to lists that work with Sankey
for key, value in link_counter.items():
    src, tgt = key
    source.append(src)
    target.append(tgt)
    values.append(value)

In [13]:
# References: https://plotly.com/python/sankey-diagram/
# Draw Sankey Diagram
fig = go.Figure(data=[go.Sankey(
    node=dict(
        pad=15,
        thickness=15,
        line=dict(color="black", width=1),
        label=labels
    ),
    link=dict(
        source=source,
        target=target,
        value=values
    ))])

fig.update_layout(title_text=f"Entity Relationships in Haunted Places Description (Top {top_N} Most Relevant Entities)", font_size=10)
fig.show()
fig.write_html("/content/drive/MyDrive/DSCI_550/entity_relationships_sankey.html")